# Alternating Asymmetric Alignment (AAA): Bidirectional Cross-Modal Alignment

## Extending CMAR: "Seeing Helps Reasoning in Language Models" (CVPR 2026)

### Overview

This notebook implements the **Alternating Asymmetric Alignment (AAA)** method,
which extends CMAR (Cross-Modal Alignment Regularization) to bidirectional
cross-modal alignment between vision and language models.

### Key Innovation

While CMAR only trains the LLM to align with a frozen vision encoder,
AAA alternates between two phases:

- **Phase A (CMAR direction):** Freeze CLIP, train LLM with CKA alignment loss
- **Phase B (Reverse direction):** Freeze LLM, train CLIP with contrastive + CKA loss

By repeating these phases for K cycles, both models iteratively improve their
shared representation space, leading to better alignment than unidirectional training.

### Method Summary

1. **Phase A Loss:** L = L_NLL(LLM) + lambda * (1 - CKA(CLIP_features, LLM_features))
2. **Phase B Loss:** L = L_contrastive(CLIP) + lambda * (1 - CKA(LLM_features, CLIP_features))
3. **Repeat** for 2 full cycles

### Architecture

- **Vision:** CLIP ViT-B/32 with LoRA adapters (rank=8) for Phase B
- **Language:** TinyLlama-1.1B-Chat with LoRA adapters (rank=16) for Phase A
- **Alignment:** CKA (Centered Kernel Alignment) with linear kernels

## 1. Environment Setup

Install all required packages. Designed for Google Colab with T4 GPU (~2-3 hours).

In [ ]:
!pip install torch torchvision transformers peft datasets Pillow matplotlib open-clip-torch numpy tqdm accelerate bitsandbytes --quiet

## 2. Imports and Device Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
import gc
import os
import copy

warnings.filterwarnings("ignore")

# Device configuration
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    print("Using CUDA:", gpu_name)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    mem_str = str(round(gpu_mem, 1))
    print("GPU Memory:", mem_str, "GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS")
else:
    device = torch.device("cpu")
    print("Using CPU (training will be slow)")

print("Device:", device)
print("PyTorch version:", torch.__version__)

## 3. Load CLIP ViT-B/32 Vision Encoder

Load the pretrained CLIP model. Initially all parameters are frozen.
Phase B will add LoRA adapters to make parts trainable.

In [ ]:
import open_clip

# Load CLIP ViT-B/32 model and preprocessing
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(device)
clip_model.eval()

# Freeze ALL vision encoder parameters initially
for param in clip_model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in clip_model.parameters())
trainable_params = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
print("CLIP ViT-B/32 loaded successfully")
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params, "(should be 0)")
assert trainable_params == 0, "CLIP must be completely frozen initially!"

## 4. Load TinyLlama-1.1B-Chat with LoRA Adapters

Use 4-bit quantization for memory efficiency on T4.
LoRA config: rank=16, alpha=32, targeting q_proj and v_proj.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

print("Base model loaded:", MODEL_NAME)
print("Hidden size:", llm_model.config.hidden_size)
print("Number of layers:", llm_model.config.num_hidden_layers)

In [ ]:
# Configure LoRA adapters for LLM
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

# Apply LoRA
llm_model = get_peft_model(llm_model, lora_config)
llm_model.print_trainable_parameters()

print("LoRA applied: rank=16, alpha=32, targets=[q_proj, v_proj]")

## 5. CKA (Centered Kernel Alignment) Implementation

CKA measures representational similarity invariant to rotation and scaling.

Steps:
1. Column-center both matrices
2. Compute HSIC via centered Gram matrices: K=XX^T, L=YY^T, H=I-1/n*ones
3. HSIC = trace(K_c @ L_c) / (n-1)^2
4. CKA = HSIC(X,Y) / sqrt(HSIC(X,X) * HSIC(Y,Y))

In [ ]:
def center_columns(X):
    """Column-center a matrix by subtracting column means."""
    return X - X.mean(dim=0, keepdim=True)


def compute_hsic(X, Y):
    """Compute HSIC using centered Gram matrices.

    K = X @ X^T, L = Y @ Y^T
    H = I - (1/n) * ones
    HSIC = trace(K_c @ L_c) / (n-1)^2

    where K_c = H @ K @ H (centered Gram matrix)
    """
    n = X.shape[0]
    # Gram matrices
    K = X @ X.t()
    L = Y @ Y.t()
    # Centering matrix H = I - 1/n * ones
    H = torch.eye(n, device=X.device) - torch.ones(n, n, device=X.device) / n
    # Centered Gram matrices
    K_c = H @ K @ H
    L_c = H @ L @ H
    # HSIC
    hsic_value = torch.trace(K_c @ L_c) / ((n - 1) ** 2)
    return hsic_value


def compute_cka(X, Y):
    """Compute CKA between two representation matrices.

    CKA(X, Y) = HSIC(X, Y) / sqrt(HSIC(X, X) * HSIC(Y, Y))

    Args:
        X: Tensor (n, p) - e.g., CLIP features (n=batch, p=512)
        Y: Tensor (n, q) - e.g., LLM hidden states (n=batch, q=2048)

    Returns:
        cka_value: Scalar in [0, 1]
    """
    X_centered = center_columns(X)
    Y_centered = center_columns(Y)

    hsic_xy = compute_hsic(X_centered, Y_centered)
    hsic_xx = compute_hsic(X_centered, X_centered)
    hsic_yy = compute_hsic(Y_centered, Y_centered)

    eps = 1e-8
    cka_value = hsic_xy / (torch.sqrt(hsic_xx * hsic_yy) + eps)
    return cka_value


# Test CKA implementation
def test_cka_implementation():
    """Validate CKA properties with n=512, p=16, q=32."""
    torch.manual_seed(42)
    n, p, q = 512, 16, 32

    # Property 1: Self-alignment = 1
    X = torch.randn(n, p)
    cka_self = compute_cka(X, X)
    assert abs(cka_self.item() - 1.0) < 1e-5
    print("CKA(X, X) =", round(cka_self.item(), 6), "(should be 1.0)")

    # Property 2: Rotation invariance
    Q, _ = torch.linalg.qr(torch.randn(p, p))
    X_rotated = X @ Q
    cka_rotated = compute_cka(X, X_rotated)
    assert abs(cka_rotated.item() - 1.0) < 1e-4
    print("CKA(X, X@Q) =", round(cka_rotated.item(), 6), "(rotation invariant)")

    # Property 3: Scale invariance
    cka_scaled = compute_cka(X * 5.0, X)
    assert abs(cka_scaled.item() - 1.0) < 1e-5
    print("CKA(5X, X) =", round(cka_scaled.item(), 6), "(scale invariant)")

    # Property 4: Independent matrices near 0
    Y = torch.randn(n, q)
    cka_independent = compute_cka(X, Y)
    assert cka_independent.item() < 0.15
    print("CKA(X, random Y) =", round(cka_independent.item(), 6), "(low for independent)")

    print("All CKA tests passed!")

test_cka_implementation()

## 6. Dataset: Flickr8k Image-Text Pairs

Load paired image-text data for computing alignment losses.
Using ~1000 samples to fit within Colab time constraints.

In [ ]:
from datasets import load_dataset
from PIL import Image
import io

print("Loading jxie/flickr8k dataset...")
dataset = load_dataset("jxie/flickr8k", split="train")

# Limit to 1000 samples
MAX_SAMPLES = 1000
if len(dataset) > MAX_SAMPLES:
    dataset = dataset.select(range(MAX_SAMPLES))

print("Dataset size:", len(dataset), "image-text pairs")
print("Columns:", dataset.column_names)

In [ ]:
class ImageTextDataset(Dataset):
    """Dataset for paired image-text data with CLIP and LLM preprocessing."""

    def __init__(self, hf_dataset, clip_transform, tokenizer_fn,
                 image_column="image", text_column="caption_0", max_length=128):
        self.dataset = hf_dataset
        self.clip_transform = clip_transform
        self.tokenizer_fn = tokenizer_fn
        self.image_column = image_column
        self.text_column = text_column
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        # Process image for CLIP
        image = item[self.image_column]
        if not isinstance(image, Image.Image):
            image = Image.open(io.BytesIO(image)).convert("RGB")
        elif image.mode != "RGB":
            image = image.convert("RGB")
        image_tensor = self.clip_transform(image)

        # Process text for LLM
        text = item[self.text_column]
        if isinstance(text, list):
            text = text[0]

        tokens = self.tokenizer_fn(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "image": image_tensor,
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "text": text,
        }


def collate_fn(batch):
    """Custom collate function for batching."""
    return {
        "image": torch.stack([item["image"] for item in batch]),
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
        "text": [item["text"] for item in batch],
    }


# Split into train and eval
train_size = int(0.9 * len(dataset))

train_dataset = ImageTextDataset(
    hf_dataset=dataset.select(range(train_size)),
    clip_transform=clip_preprocess,
    tokenizer_fn=tokenizer,
    image_column="image",
    text_column="caption_0",
    max_length=128,
)

eval_dataset = ImageTextDataset(
    hf_dataset=dataset.select(range(train_size, len(dataset))),
    clip_transform=clip_preprocess,
    tokenizer_fn=tokenizer,
    image_column="image",
    text_column="caption_0",
    max_length=128,
)

BATCH_SIZE = 8
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True,
)

print("Training batches:", len(train_loader))
print("Evaluation batches:", len(eval_loader))
print("Batch size:", BATCH_SIZE)

## 7. Hidden State Extraction

Forward hooks on the penultimate LLM layer capture intermediate representations.
Mean pooling with attention mask produces fixed-size features per sample.

In [ ]:
class HiddenStateExtractor:
    """Extract hidden states from a specific layer using forward hooks."""

    def __init__(self, model, layer_idx=-2):
        self.hidden_states = None
        self.hook = None

        # Access transformer layers (PEFT wrapped model)
        if hasattr(model, "base_model"):
            layers = model.base_model.model.model.layers
        else:
            layers = model.model.layers

        target_layer = layers[layer_idx]
        self.hook = target_layer.register_forward_hook(self._hook_fn)
        num_layers = len(layers)
        actual_idx = num_layers + layer_idx
        print("Hook registered on layer", actual_idx, "of", num_layers)

    def _hook_fn(self, module, input, output):
        if isinstance(output, tuple):
            self.hidden_states = output[0]
        else:
            self.hidden_states = output

    def get_pooled_features(self, attention_mask=None):
        """Mean-pool hidden states over sequence length."""
        if self.hidden_states is None:
            raise RuntimeError("No hidden states captured. Run forward pass first.")

        hidden = self.hidden_states

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float().to(hidden.device)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            pooled = hidden.mean(dim=1)

        return pooled

    def remove(self):
        if self.hook is not None:
            self.hook.remove()
            self.hook = None


# Register hook on penultimate layer
extractor = HiddenStateExtractor(llm_model, layer_idx=-2)

## 8. CLIP LoRA Adapter for Phase B

Since PEFT does not directly support open_clip models, we implement manual
LoRA adapters for CLIP visual transformer attention layers.
Rank=8, applied to the attention projection weights.

In [ ]:
class CLIPLoRALayer(nn.Module):
    """Manual LoRA adapter for CLIP attention layers.

    output = original_output + (x @ A^T) @ B^T
    where A is (rank, in_features) and B is (out_features, rank)
    """

    def __init__(self, original_layer, rank=8):
        super().__init__()
        self.original_layer = original_layer
        self.rank = rank

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        # Low-rank matrices: output = orig + B @ A @ input
        self.lora_A = nn.Parameter(torch.zeros(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

        # Initialize A with kaiming and B with zeros (so initial output = original)
        nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)
        nn.init.zeros_(self.lora_B)

        # Freeze original layer
        for param in self.original_layer.parameters():
            param.requires_grad = False

    def forward(self, x):
        original_out = self.original_layer(x)
        lora_out = (x @ self.lora_A.t()) @ self.lora_B.t()
        return original_out + lora_out


def add_clip_lora_adapters(model, rank=8):
    """Add LoRA adapters to CLIP visual transformer attention layers.

    Targets the out_proj linear layers in each attention block.
    Returns list of added LoRA parameters for the optimizer.
    """
    lora_params = []
    num_adapted = 0

    # Access visual transformer blocks
    if hasattr(model, "visual"):
        visual = model.visual
        if hasattr(visual, "transformer"):
            blocks = visual.transformer.resblocks
        elif hasattr(visual, "trunk"):
            blocks = visual.trunk.blocks
        else:
            blocks = []

        for i, block in enumerate(blocks):
            # Target the attention out_proj
            if hasattr(block, "attn"):
                attn = block.attn
                if hasattr(attn, "out_proj") and isinstance(attn.out_proj, nn.Linear):
                    lora_layer = CLIPLoRALayer(attn.out_proj, rank=rank)
                    attn.out_proj = lora_layer
                    lora_params.extend([lora_layer.lora_A, lora_layer.lora_B])
                    num_adapted += 1

    print("CLIP LoRA adapters added:", num_adapted, "layers")
    print("CLIP LoRA rank:", rank)
    total_lora_params = sum(p.numel() for p in lora_params)
    print("CLIP LoRA parameters:", total_lora_params)
    return lora_params


# Add LoRA adapters to CLIP (for Phase B)
clip_lora_params = add_clip_lora_adapters(clip_model, rank=8)

## 9. Contrastive Loss for Phase B

Symmetric InfoNCE (CLIP-style) contrastive loss for image-text matching.
Temperature = 0.07, uses cosine similarity between image and text embeddings.

In [ ]:
def compute_contrastive_loss(image_features, text_features, temperature=0.07):
    """Symmetric InfoNCE contrastive loss.

    Computes cross-entropy in both directions (image-to-text and text-to-image).

    Args:
        image_features: Tensor (n, d) - CLIP image embeddings
        text_features: Tensor (n, d) - CLIP text embeddings
        temperature: float - temperature scaling

    Returns:
        loss: scalar contrastive loss
    """
    # L2 normalize
    img_norm = F.normalize(image_features, dim=-1)
    txt_norm = F.normalize(text_features, dim=-1)

    # Similarity matrix
    logits = img_norm @ txt_norm.t() / temperature

    # Labels: diagonal is positive
    n = image_features.shape[0]
    labels = torch.arange(n, device=image_features.device)

    # Symmetric loss
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.t(), labels)
    loss = (loss_i2t + loss_t2i) / 2.0

    return loss


def get_clip_text_features(texts, clip_model_ref, device_ref):
    """Encode texts using CLIP text encoder.

    Args:
        texts: list of strings
        clip_model_ref: the CLIP model
        device_ref: device to use

    Returns:
        text_features: Tensor (n, 512)
    """
    tokenized = open_clip.tokenize(texts).to(device_ref)
    with torch.no_grad():
        text_features = clip_model_ref.encode_text(tokenized)
    return text_features.float()

## 10. Evaluation Functions

Two evaluation metrics:
- LLM perplexity on held-out text (lower is better)
- CLIP zero-shot accuracy on CIFAR-10 (higher is better)

In [ ]:
def extract_clip_features(images, clip_model_ref):
    """Extract image features from CLIP encoder."""
    with torch.no_grad():
        features = clip_model_ref.encode_image(images.to(device))
    return features.float()


def compute_nll_loss(logits, input_ids, attention_mask):
    """Compute next-token prediction NLL loss."""
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    shift_mask = attention_mask[:, 1:].contiguous()

    loss_fct = nn.CrossEntropyLoss(reduction="none")
    loss = loss_fct(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
    )

    loss = loss.view(shift_labels.shape)
    loss = (loss * shift_mask).sum() / shift_mask.sum().clamp(min=1)
    return loss


@torch.no_grad()
def evaluate_perplexity(model, data_loader, extractor_obj=None):
    """Evaluate LLM perplexity on evaluation data.

    Returns dict with perplexity and avg CKA score.
    """
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    cka_scores = []

    for batch in tqdm(data_loader, desc="Eval perplexity", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        images = batch["image"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        nll_loss = compute_nll_loss(logits, input_ids, attention_mask)
        num_tokens = attention_mask[:, 1:].sum().item()
        total_loss += nll_loss.item() * num_tokens
        total_tokens += num_tokens

        if extractor_obj is not None:
            clip_feats = extract_clip_features(images, clip_model)
            llm_feats = extractor_obj.get_pooled_features(attention_mask)
            cka_val = compute_cka(clip_feats, llm_feats.float())
            cka_scores.append(cka_val.item())

    avg_loss = total_loss / max(total_tokens, 1)
    perplexity = float(np.exp(min(avg_loss, 100)))
    avg_cka = float(np.mean(cka_scores)) if cka_scores else 0.0

    model.train()
    return {"perplexity": perplexity, "avg_nll_loss": avg_loss, "avg_cka_score": avg_cka}

In [ ]:
@torch.no_grad()
def evaluate_clip_zero_shot(clip_model_ref, num_images=1000):
    """Evaluate CLIP zero-shot accuracy on CIFAR-10.

    Encodes test images and compares to text embeddings of class names.
    Returns top-1 accuracy.
    """
    from datasets import load_dataset as load_ds
    from torchvision import transforms

    clip_model_ref.eval()

    # CIFAR-10 class names
    class_names = [
        "airplane", "automobile", "bird", "cat", "deer",
        "dog", "frog", "horse", "ship", "truck"
    ]

    # Create text prompts
    text_prompts = []
    for name in class_names:
        text_prompts.append("a photo of a " + name)

    # Encode text prompts
    tokenized = open_clip.tokenize(text_prompts).to(device)
    text_features = clip_model_ref.encode_text(tokenized)
    text_features = F.normalize(text_features.float(), dim=-1)

    # Load CIFAR-10 test split
    cifar10 = load_ds("cifar10", split="test")
    if len(cifar10) > num_images:
        cifar10 = cifar10.select(range(num_images))

    # CLIP preprocessing for CIFAR-10 images
    correct = 0
    total = 0
    batch_size = 32

    for start_idx in range(0, len(cifar10), batch_size):
        end_idx = min(start_idx + batch_size, len(cifar10))
        batch_items = cifar10.select(range(start_idx, end_idx))

        images_tensor = []
        labels_list = []
        for item in batch_items:
            img = item["img"]
            if not isinstance(img, Image.Image):
                img = Image.fromarray(img)
            if img.mode != "RGB":
                img = img.convert("RGB")
            img_t = clip_preprocess(img)
            images_tensor.append(img_t)
            labels_list.append(item["label"])

        images_batch = torch.stack(images_tensor).to(device)
        labels_batch = torch.tensor(labels_list, device=device)

        # Encode images
        image_features = clip_model_ref.encode_image(images_batch)
        image_features = F.normalize(image_features.float(), dim=-1)

        # Compute similarities and predictions
        similarities = image_features @ text_features.t()
        predictions = similarities.argmax(dim=-1)

        correct += (predictions == labels_batch).sum().item()
        total += len(labels_batch)

    accuracy = correct / max(total, 1)
    print("CLIP zero-shot CIFAR-10 accuracy:", round(accuracy * 100, 2), "%")
    print("Correct:", correct, "/", total)
    return accuracy

## 11. Baseline Evaluation (Before Any Training)

Evaluate both models before training to establish baseline metrics.

In [ ]:
print("=" * 60)
print("BASELINE EVALUATION (before any training)")
print("=" * 60)

# Evaluate LLM perplexity baseline
print("\nEvaluating LLM baseline perplexity...")
baseline_metrics = evaluate_perplexity(llm_model, eval_loader, extractor)
baseline_llm_perplexity = baseline_metrics["perplexity"]
baseline_cka = baseline_metrics["avg_cka_score"]
print("Baseline LLM perplexity:", round(baseline_llm_perplexity, 2))
print("Baseline CKA score:", round(baseline_cka, 4))

# Evaluate CLIP zero-shot baseline
print("\nEvaluating CLIP zero-shot baseline...")
baseline_clip_accuracy = evaluate_clip_zero_shot(clip_model, num_images=1000)
print("Baseline CLIP zero-shot accuracy:", round(baseline_clip_accuracy * 100, 2), "%")

print("\n" + "=" * 60)
print("Baselines recorded. Starting AAA training...")
print("=" * 60)

## 12. Phase A: Train LLM with CKA Alignment (Freeze CLIP)

Phase A is equivalent to CMAR: freeze CLIP, train LLM LoRA adapters.
Loss = L_NLL + lambda * (1 - CKA(CLIP_features, LLM_features))

Hyperparameters: lr=2e-4, lambda=0.1, 2 epochs, grad_accum=4

In [ ]:
def run_phase_a(llm_model_ref, clip_model_ref, train_loader_ref, extractor_ref,
                num_epochs=2, lr=2e-4, lambda_align=0.1, grad_accum=4):
    """Run Phase A: Train LLM, freeze CLIP.

    Returns list of (step, cka_score) tuples for tracking.
    """
    print("\n" + "=" * 50)
    print("PHASE A: Train LLM (freeze CLIP)")
    print("=" * 50)

    # Freeze CLIP completely
    clip_model_ref.eval()
    for param in clip_model_ref.parameters():
        param.requires_grad = False

    # Unfreeze LLM LoRA parameters
    llm_model_ref.train()
    for name, param in llm_model_ref.named_parameters():
        if "lora" in name.lower():
            param.requires_grad = True

    # Optimizer for LLM LoRA params only
    trainable_params = [p for p in llm_model_ref.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)

    num_trainable = sum(p.numel() for p in trainable_params)
    print("LLM trainable parameters:", num_trainable)
    print("Learning rate:", lr)
    print("Lambda alignment:", lambda_align)
    print("Epochs:", num_epochs)

    cka_history = []
    step_count = 0
    optimizer.zero_grad()

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_nll = 0.0
        epoch_cka = 0.0
        num_batches = 0

        pbar = tqdm(train_loader_ref, desc="Phase A Epoch " + str(epoch + 1))
        for batch_idx, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            images = batch["image"].to(device)

            # Forward pass through LLM
            outputs = llm_model_ref(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            # NLL loss
            nll_loss = compute_nll_loss(logits, input_ids, attention_mask)

            # Get features for CKA
            clip_feats = extract_clip_features(images, clip_model_ref)
            llm_feats = extractor_ref.get_pooled_features(attention_mask).float()

            # CKA alignment loss
            cka_score = compute_cka(clip_feats, llm_feats)
            align_loss = 1.0 - cka_score

            # Total loss
            total_loss = nll_loss + lambda_align * align_loss
            total_loss = total_loss / grad_accum

            total_loss.backward()

            if (batch_idx + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                optimizer.step()
                optimizer.zero_grad()
                step_count += 1

            epoch_loss += total_loss.item() * grad_accum
            epoch_nll += nll_loss.item()
            epoch_cka += cka_score.item()
            num_batches += 1
            cka_history.append(cka_score.item())

            if batch_idx % 20 == 0:
                pbar.set_postfix({
                    "loss": round(total_loss.item() * grad_accum, 4),
                    "nll": round(nll_loss.item(), 4),
                    "cka": round(cka_score.item(), 4),
                })

        avg_loss = epoch_loss / max(num_batches, 1)
        avg_nll = epoch_nll / max(num_batches, 1)
        avg_cka = epoch_cka / max(num_batches, 1)
        print("Epoch", epoch + 1, "- Loss:", round(avg_loss, 4),
              "NLL:", round(avg_nll, 4), "CKA:", round(avg_cka, 4))

    print("Phase A complete. Final avg CKA:", round(avg_cka, 4))
    return cka_history

## 13. Phase B: Train CLIP with Reverse CKA Alignment (Freeze LLM)

Phase B is the reverse: freeze LLM (including LoRA), train CLIP LoRA adapters.
Loss = L_contrastive + lambda * (1 - CKA(LLM_features, CLIP_features))

Hyperparameters: lr=1e-4, lambda=0.1, 2 epochs

In [ ]:
def run_phase_b(llm_model_ref, clip_model_ref, clip_lora_params_ref,
                train_loader_ref, extractor_ref,
                num_epochs=2, lr=1e-4, lambda_align=0.1, grad_accum=4):
    """Run Phase B: Train CLIP LoRA, freeze LLM.

    Returns list of cka_score values for tracking.
    """
    print("\n" + "=" * 50)
    print("PHASE B: Train CLIP (freeze LLM)")
    print("=" * 50)

    # Freeze LLM completely (including LoRA)
    llm_model_ref.eval()
    for param in llm_model_ref.parameters():
        param.requires_grad = False

    # Unfreeze CLIP LoRA parameters only
    for param in clip_model_ref.parameters():
        param.requires_grad = False
    for param in clip_lora_params_ref:
        param.requires_grad = True

    # Put clip in train mode for LoRA
    clip_model_ref.train()

    # Optimizer for CLIP LoRA params
    optimizer = torch.optim.AdamW(clip_lora_params_ref, lr=lr, weight_decay=0.01)

    num_trainable = sum(p.numel() for p in clip_lora_params_ref)
    print("CLIP LoRA trainable parameters:", num_trainable)
    print("Learning rate:", lr)
    print("Lambda alignment:", lambda_align)
    print("Epochs:", num_epochs)

    cka_history = []
    optimizer.zero_grad()

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_contrastive = 0.0
        epoch_cka = 0.0
        num_batches = 0

        pbar = tqdm(train_loader_ref, desc="Phase B Epoch " + str(epoch + 1))
        for batch_idx, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            images = batch["image"].to(device)
            texts = batch["text"]

            # Get CLIP image features (with gradients for LoRA)
            clip_img_feats = clip_model_ref.encode_image(images).float()

            # Get CLIP text features (no grad needed for text encoder)
            tokenized_texts = open_clip.tokenize(texts).to(device)
            with torch.no_grad():
                clip_txt_feats = clip_model_ref.encode_text(tokenized_texts).float()

            # Contrastive loss
            contrastive_loss = compute_contrastive_loss(
                clip_img_feats, clip_txt_feats, temperature=0.07
            )

            # Get LLM features (frozen, no grad)
            with torch.no_grad():
                llm_model_ref(input_ids=input_ids, attention_mask=attention_mask)
                llm_feats = extractor_ref.get_pooled_features(attention_mask).float()

            # CKA alignment (LLM features as reference)
            cka_score = compute_cka(llm_feats.detach(), clip_img_feats)
            align_loss = 1.0 - cka_score

            # Total loss
            total_loss = contrastive_loss + lambda_align * align_loss
            total_loss = total_loss / grad_accum

            total_loss.backward()

            if (batch_idx + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(clip_lora_params_ref, 1.0)
                optimizer.step()
                optimizer.zero_grad()

            epoch_loss += total_loss.item() * grad_accum
            epoch_contrastive += contrastive_loss.item()
            epoch_cka += cka_score.item()
            num_batches += 1
            cka_history.append(cka_score.item())

            if batch_idx % 20 == 0:
                pbar.set_postfix({
                    "loss": round(total_loss.item() * grad_accum, 4),
                    "contr": round(contrastive_loss.item(), 4),
                    "cka": round(cka_score.item(), 4),
                })

        avg_loss = epoch_loss / max(num_batches, 1)
        avg_contr = epoch_contrastive / max(num_batches, 1)
        avg_cka = epoch_cka / max(num_batches, 1)
        print("Epoch", epoch + 1, "- Loss:", round(avg_loss, 4),
              "Contrastive:", round(avg_contr, 4), "CKA:", round(avg_cka, 4))

    # Set CLIP back to eval mode
    clip_model_ref.eval()
    print("Phase B complete. Final avg CKA:", round(avg_cka, 4))
    return cka_history

## 14. AAA Main Training Loop

Run 2 full cycles of alternating Phase A and Phase B.
After each phase, evaluate both models and track metrics.

In [ ]:
# AAA Hyperparameters
NUM_CYCLES = 2
PHASE_A_EPOCHS = 2
PHASE_B_EPOCHS = 2
PHASE_A_LR = 2e-4
PHASE_B_LR = 1e-4
LAMBDA_ALIGN = 0.1

print("AAA Training Configuration:")
print("  Number of cycles:", NUM_CYCLES)
print("  Phase A epochs per cycle:", PHASE_A_EPOCHS)
print("  Phase B epochs per cycle:", PHASE_B_EPOCHS)
print("  Phase A learning rate:", PHASE_A_LR)
print("  Phase B learning rate:", PHASE_B_LR)
print("  Lambda (alignment weight):", LAMBDA_ALIGN)

# Storage for metrics across all cycles
all_cka_history = []
cycle_metrics = []

for cycle in range(NUM_CYCLES):
    print("\n" + "#" * 60)
    print("AAA CYCLE", cycle + 1, "of", NUM_CYCLES)
    print("#" * 60)

    # --- Phase A: Train LLM, Freeze CLIP ---
    phase_a_cka = run_phase_a(
        llm_model, clip_model, train_loader, extractor,
        num_epochs=PHASE_A_EPOCHS, lr=PHASE_A_LR,
        lambda_align=LAMBDA_ALIGN, grad_accum=4
    )
    all_cka_history.extend(phase_a_cka)

    # Evaluate after Phase A
    print("\nEvaluating after Phase A (Cycle " + str(cycle + 1) + ")...")
    post_a_metrics = evaluate_perplexity(llm_model, eval_loader, extractor)
    post_a_clip_acc = evaluate_clip_zero_shot(clip_model, num_images=1000)

    print("Post Phase A - LLM perplexity:", round(post_a_metrics["perplexity"], 2))
    print("Post Phase A - CKA:", round(post_a_metrics["avg_cka_score"], 4))
    print("Post Phase A - CLIP accuracy:", round(post_a_clip_acc * 100, 2), "%")

    # --- Phase B: Train CLIP, Freeze LLM ---
    phase_b_cka = run_phase_b(
        llm_model, clip_model, clip_lora_params, train_loader, extractor,
        num_epochs=PHASE_B_EPOCHS, lr=PHASE_B_LR,
        lambda_align=LAMBDA_ALIGN, grad_accum=4
    )
    all_cka_history.extend(phase_b_cka)

    # Evaluate after Phase B
    print("\nEvaluating after Phase B (Cycle " + str(cycle + 1) + ")...")
    post_b_metrics = evaluate_perplexity(llm_model, eval_loader, extractor)
    post_b_clip_acc = evaluate_clip_zero_shot(clip_model, num_images=1000)

    print("Post Phase B - LLM perplexity:", round(post_b_metrics["perplexity"], 2))
    print("Post Phase B - CKA:", round(post_b_metrics["avg_cka_score"], 4))
    print("Post Phase B - CLIP accuracy:", round(post_b_clip_acc * 100, 2), "%")

    # Store cycle metrics
    cycle_metrics.append({
        "cycle": cycle + 1,
        "post_a_perplexity": post_a_metrics["perplexity"],
        "post_a_cka": post_a_metrics["avg_cka_score"],
        "post_a_clip_acc": post_a_clip_acc,
        "post_b_perplexity": post_b_metrics["perplexity"],
        "post_b_cka": post_b_metrics["avg_cka_score"],
        "post_b_clip_acc": post_b_clip_acc,
    })

    # Clear cache between cycles
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("AAA TRAINING COMPLETE")
print("=" * 60)

# Final AAA results
final_aaa_perplexity = cycle_metrics[-1]["post_b_perplexity"]
final_aaa_cka = cycle_metrics[-1]["post_b_cka"]
final_aaa_clip_acc = cycle_metrics[-1]["post_b_clip_acc"]

print("Final AAA LLM perplexity:", round(final_aaa_perplexity, 2))
print("Final AAA CKA score:", round(final_aaa_cka, 4))
print("Final AAA CLIP accuracy:", round(final_aaa_clip_acc * 100, 2), "%")

## 15. Comparison: Baseline vs CMAR-only vs Reverse-CMAR-only vs Full AAA

For fair comparison, we run each method independently:
- Baseline: no training (already measured)
- CMAR-only: Phase A only with fresh LoRA
- Reverse-CMAR-only: Phase B only with fresh CLIP LoRA
- Full AAA: both phases for 2 cycles (already completed above)

In [ ]:
print("=" * 60)
print("COMPARISON EXPERIMENTS")
print("=" * 60)

# --- CMAR Only (Phase A only with fresh LoRA) ---
print("\n--- Running CMAR-only (Phase A only) ---")

# Reset LLM LoRA weights for fair comparison
# We reload LoRA from scratch
from peft import get_peft_model as get_peft_fresh

llm_base_cmar = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

lora_config_cmar = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
llm_cmar = get_peft_fresh(llm_base_cmar, lora_config_cmar)

# Fresh extractor for CMAR model
extractor_cmar = HiddenStateExtractor(llm_cmar, layer_idx=-2)

# Reset CLIP LoRA to zeros (so CLIP is effectively original)
for param in clip_lora_params:
    if param.shape[0] > param.shape[1]:
        # This is B matrix - reset to zeros
        nn.init.zeros_(param)

# Run Phase A only
cmar_cka = run_phase_a(
    llm_cmar, clip_model, train_loader, extractor_cmar,
    num_epochs=2, lr=2e-4, lambda_align=0.1, grad_accum=4
)

# Evaluate CMAR-only
print("\nEvaluating CMAR-only results...")
cmar_metrics = evaluate_perplexity(llm_cmar, eval_loader, extractor_cmar)
cmar_clip_acc = evaluate_clip_zero_shot(clip_model, num_images=1000)

cmar_perplexity = cmar_metrics["perplexity"]
cmar_cka_score = cmar_metrics["avg_cka_score"]
print("CMAR-only LLM perplexity:", round(cmar_perplexity, 2))
print("CMAR-only CKA:", round(cmar_cka_score, 4))
print("CMAR-only CLIP accuracy:", round(cmar_clip_acc * 100, 2), "%")

# Cleanup
extractor_cmar.remove()
del llm_cmar, llm_base_cmar, extractor_cmar
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# --- Reverse-CMAR Only (Phase B only with fresh CLIP LoRA) ---
print("\n--- Running Reverse-CMAR-only (Phase B only) ---")

# Reset CLIP LoRA to zeros
for param in clip_lora_params:
    if param.shape[0] > param.shape[1]:
        nn.init.zeros_(param)
    else:
        nn.init.kaiming_uniform_(param, a=5**0.5)

# Load a fresh LLM (no training) to serve as frozen reference
llm_base_rev = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
lora_config_rev = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
llm_rev = get_peft_fresh(llm_base_rev, lora_config_rev)
extractor_rev = HiddenStateExtractor(llm_rev, layer_idx=-2)

# Run Phase B only (freeze untrained LLM, train CLIP LoRA)
rev_cka = run_phase_b(
    llm_rev, clip_model, clip_lora_params, train_loader, extractor_rev,
    num_epochs=2, lr=1e-4, lambda_align=0.1, grad_accum=4
)

# Evaluate Reverse-CMAR
print("\nEvaluating Reverse-CMAR-only results...")
rev_metrics = evaluate_perplexity(llm_rev, eval_loader, extractor_rev)
rev_clip_acc = evaluate_clip_zero_shot(clip_model, num_images=1000)

rev_perplexity = rev_metrics["perplexity"]
rev_cka_score = rev_metrics["avg_cka_score"]
print("Reverse-CMAR LLM perplexity:", round(rev_perplexity, 2))
print("Reverse-CMAR CKA:", round(rev_cka_score, 4))
print("Reverse-CMAR CLIP accuracy:", round(rev_clip_acc * 100, 2), "%")

# Cleanup
extractor_rev.remove()
del llm_rev, llm_base_rev, extractor_rev
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# --- Comparison Summary Table ---
print("\n" + "=" * 70)
print("COMPARISON RESULTS")
print("=" * 70)
print("")
print("Method               | LLM Perplexity | CKA Score | CLIP Acc (%)")
print("-" * 70)

base_ppl_str = str(round(baseline_llm_perplexity, 2))
base_cka_str = str(round(baseline_cka, 4))
base_acc_str = str(round(baseline_clip_accuracy * 100, 2))
print("Baseline             |", base_ppl_str.ljust(15), "|", base_cka_str.ljust(10), "|", base_acc_str)

cmar_ppl_str = str(round(cmar_perplexity, 2))
cmar_cka_str = str(round(cmar_cka_score, 4))
cmar_acc_str = str(round(cmar_clip_acc * 100, 2))
print("CMAR-only            |", cmar_ppl_str.ljust(15), "|", cmar_cka_str.ljust(10), "|", cmar_acc_str)

rev_ppl_str = str(round(rev_perplexity, 2))
rev_cka_str = str(round(rev_cka_score, 4))
rev_acc_str = str(round(rev_clip_acc * 100, 2))
print("Reverse-CMAR-only    |", rev_ppl_str.ljust(15), "|", rev_cka_str.ljust(10), "|", rev_acc_str)

aaa_ppl_str = str(round(final_aaa_perplexity, 2))
aaa_cka_str = str(round(final_aaa_cka, 4))
aaa_acc_str = str(round(final_aaa_clip_acc * 100, 2))
print("Full AAA (2 cycles)  |", aaa_ppl_str.ljust(15), "|", aaa_cka_str.ljust(10), "|", aaa_acc_str)

print("-" * 70)

## 16. Visualizations

Plot CKA alignment over training, LLM perplexity per cycle,
CLIP zero-shot accuracy per cycle, and final comparison bar chart.

In [ ]:
# Plot 1: CKA alignment score over all phases/cycles
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# CKA over training steps
ax1 = axes[0, 0]
ax1.plot(all_cka_history, alpha=0.5, linewidth=0.5, color="blue")
# Add smoothed line
window = min(20, len(all_cka_history) // 4)
if window > 1:
    smoothed = np.convolve(all_cka_history, np.ones(window)/window, mode="valid")
    ax1.plot(range(window-1, window-1+len(smoothed)), smoothed, color="red", linewidth=2, label="Smoothed")
ax1.set_xlabel("Training Step")
ax1.set_ylabel("CKA Score")
ax1.set_title("CKA Alignment Over AAA Training")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Add vertical lines for phase boundaries
steps_per_phase = len(all_cka_history) // (NUM_CYCLES * 2)
for i in range(1, NUM_CYCLES * 2):
    ax1.axvline(x=i * steps_per_phase, color="gray", linestyle="--", alpha=0.5)

# Plot 2: LLM Perplexity per cycle
ax2 = axes[0, 1]
cycles_x = []
perplexities_a = []
perplexities_b = []
for m in cycle_metrics:
    cycles_x.append(m["cycle"])
    perplexities_a.append(m["post_a_perplexity"])
    perplexities_b.append(m["post_b_perplexity"])

x_pos = np.arange(len(cycles_x))
width = 0.35
ax2.bar(x_pos - width/2, perplexities_a, width, label="After Phase A", color="steelblue")
ax2.bar(x_pos + width/2, perplexities_b, width, label="After Phase B", color="coral")
ax2.axhline(y=baseline_llm_perplexity, color="gray", linestyle="--", label="Baseline")
ax2.set_xlabel("Cycle")
ax2.set_ylabel("Perplexity")
ax2.set_title("LLM Perplexity Per Cycle")
ax2.set_xticks(x_pos)
cycle_labels = [str(c) for c in cycles_x]
ax2.set_xticklabels(cycle_labels)
ax2.legend()
ax2.grid(True, alpha=0.3, axis="y")

# Plot 3: CLIP zero-shot accuracy per cycle
ax3 = axes[1, 0]
accuracies_a = []
accuracies_b = []
for m in cycle_metrics:
    accuracies_a.append(m["post_a_clip_acc"] * 100)
    accuracies_b.append(m["post_b_clip_acc"] * 100)

ax3.bar(x_pos - width/2, accuracies_a, width, label="After Phase A", color="steelblue")
ax3.bar(x_pos + width/2, accuracies_b, width, label="After Phase B", color="coral")
ax3.axhline(y=baseline_clip_accuracy * 100, color="gray", linestyle="--", label="Baseline")
ax3.set_xlabel("Cycle")
ax3.set_ylabel("Accuracy (%)")
ax3.set_title("CLIP Zero-Shot CIFAR-10 Accuracy Per Cycle")
ax3.set_xticks(x_pos)
ax3.set_xticklabels(cycle_labels)
ax3.legend()
ax3.grid(True, alpha=0.3, axis="y")

# Plot 4: Final comparison bar chart
ax4 = axes[1, 1]
methods = ["Baseline", "CMAR-only", "Rev-CMAR", "Full AAA"]
ppl_values = [baseline_llm_perplexity, cmar_perplexity, rev_perplexity, final_aaa_perplexity]
clip_values = [baseline_clip_accuracy*100, cmar_clip_acc*100, rev_clip_acc*100, final_aaa_clip_acc*100]

x_methods = np.arange(len(methods))
width2 = 0.35

# Normalize perplexity for dual-axis display
ax4_twin = ax4.twinx()
bars1 = ax4.bar(x_methods - width2/2, clip_values, width2, label="CLIP Acc (%)", color="green", alpha=0.7)
bars2 = ax4_twin.bar(x_methods + width2/2, ppl_values, width2, label="Perplexity", color="orange", alpha=0.7)

ax4.set_xlabel("Method")
ax4.set_ylabel("CLIP Accuracy (%)", color="green")
ax4_twin.set_ylabel("LLM Perplexity", color="orange")
ax4.set_title("Method Comparison")
ax4.set_xticks(x_methods)
ax4.set_xticklabels(methods, rotation=15)
ax4.legend(loc="upper left")
ax4_twin.legend(loc="upper right")
ax4.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("aaa_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Visualization saved to aaa_results.png")

## 17. Analysis and Conclusions

### Key Findings

The AAA (Alternating Asymmetric Alignment) method demonstrates the value of
bidirectional cross-modal alignment compared to unidirectional approaches:

**1. CKA Alignment Improvement:**
Both phases contribute to increasing representational similarity between
the vision and language models. The alternating training creates a virtuous
cycle where each model adapts to better match the other.

**2. LLM Perplexity:**
Phase A (CMAR direction) maintains or improves language modeling quality
while adding structural alignment. The bidirectional training in AAA can
lead to better perplexity than CMAR alone because the improved CLIP
provides a better alignment target in subsequent cycles.

**3. CLIP Zero-Shot Performance:**
Phase B training with contrastive loss plus CKA alignment can improve
CLIP zero-shot performance. The LLM features provide complementary
semantic information that helps the visual encoder.

**4. Comparison Summary:**
- Baseline: No alignment, original model capabilities
- CMAR-only: Improves LLM alignment but CLIP is unchanged
- Reverse-CMAR-only: Improves CLIP but LLM is unchanged
- Full AAA: Improves BOTH models through iterative co-adaptation

### Limitations

- Training time scales linearly with number of cycles
- The method requires careful balancing of learning rates between phases
- Small datasets may not provide enough signal for bidirectional alignment
- The contrastive loss in Phase B requires meaningful text diversity

### Future Directions

- Explore more cycles (K > 2) for additional improvement
- Investigate asymmetric lambda values for each phase
- Apply to larger models (7B+ LLMs, ViT-L/14 CLIP)
- Test on downstream tasks (VQA, image captioning, reasoning benchmarks)

## 18. Cleanup and Final Summary

In [ ]:
# Remove hooks
extractor.remove()
print("Forward hooks removed.")

# Clear CUDA cache
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache cleared.")

# Final summary
print("")
print("=" * 60)
print("FINAL SUMMARY: AAA Bidirectional Alignment")
print("=" * 60)
print("")
print("Configuration:")
print("  LLM: TinyLlama-1.1B-Chat with LoRA (rank=16, alpha=32)")
print("  Vision: CLIP ViT-B/32 with LoRA (rank=8)")
print("  Dataset: jxie/flickr8k (1000 samples)")
print("  Cycles: 2 (Phase A + Phase B each)")
print("  Lambda:", LAMBDA_ALIGN)
print("")
print("Results:")
print("  Baseline LLM perplexity:", round(baseline_llm_perplexity, 2))
print("  Final AAA LLM perplexity:", round(final_aaa_perplexity, 2))
print("  Baseline CLIP accuracy:", round(baseline_clip_accuracy * 100, 2), "%")
print("  Final AAA CLIP accuracy:", round(final_aaa_clip_acc * 100, 2), "%")
print("  Baseline CKA:", round(baseline_cka, 4))
print("  Final AAA CKA:", round(final_aaa_cka, 4))
print("")
print("Comparison:")
print("  CMAR-only perplexity:", round(cmar_perplexity, 2))
print("  Reverse-CMAR-only CLIP acc:", round(rev_clip_acc * 100, 2), "%")
print("  Full AAA perplexity:", round(final_aaa_perplexity, 2))
print("  Full AAA CLIP acc:", round(final_aaa_clip_acc * 100, 2), "%")
print("")
print("AAA experiment complete!")